# Creator Console — Colab başlatıcı

Bu notebook yereldeki Creator Console kaynak paketini Google Drive'dan açar. Proje durumu `MyDrive/YouTubeAutomation/projects`, tamamlanan arşivler `MyDrive/YouTubeAutomation/archive` altında kalır. Console yalnızca `127.0.0.1` üzerinde çalışır ve Colab'ın kullanıcı oturumuyla korunan iframe proxy'sinde gösterilir; herkese açık tünel oluşturulmaz.

Önemli: Büyük videoları iframe içindeki tarayıcı yüklemesinden geçirmek yerine Drive'daki `YouTubeAutomation/source_videos` klasörüne koyun. Colab Drive erişimini her yeni runtime'da siz onaylarsınız.

In [ ]:
# 1) Drive'ı bağla ve sabit yolları hazırla.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/YouTubeAutomation')
APP_DRIVE = DRIVE_ROOT / 'app'
BUNDLE = APP_DRIVE / 'creator_console_colab.zip'
PROJECTS = DRIVE_ROOT / 'projects'
ARCHIVE = DRIVE_ROOT / 'archive'
SOURCE_VIDEOS = DRIVE_ROOT / 'source_videos'
for folder in (APP_DRIVE, PROJECTS, ARCHIVE, SOURCE_VIDEOS):
    folder.mkdir(parents=True, exist_ok=True)
if not BUNDLE.is_file():
    raise FileNotFoundError(f'Kaynak paketi bulunamadı: {BUNDLE}')
print('Drive hazır; kaynak paketi bulundu.')

In [ ]:
# 2) Paketi geçici hızlı diske aç ve her dosyanın hash'ini doğrula.
import hashlib, json, shutil, zipfile

APP_ROOT = Path('/content/youtube_automation_app')
if APP_ROOT.exists():
    shutil.rmtree(APP_ROOT)
APP_ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE, 'r') as zf:
    names = set(zf.namelist())
    manifest = json.loads(zf.read('bundle_manifest.json'))
    for record in manifest['files']:
        rel = Path(record['path'])
        if rel.is_absolute() or '..' in rel.parts or record['path'] not in names:
            raise RuntimeError(f'Güvensiz/eksik paket girdisi: {record["path"]}')
        payload = zf.read(record['path'])
        if len(payload) != record['size'] or hashlib.sha256(payload).hexdigest() != record['sha256']:
            raise RuntimeError(f'Paket doğrulaması başarısız: {record["path"]}')
    zf.extractall(APP_ROOT)
print(f'{len(manifest["files"])} dosya doğrulanıp açıldı.')

In [ ]:
# 3) Colab kernel paketlerini bozmadan izole çalışma ortamını kur.
import subprocess, sys

VENV = Path('/content/yta_console_venv')
if not (VENV / 'bin/python').is_file():
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
PYTHON = VENV / 'bin/python'
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip'], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--quiet', '-r',
                str(APP_ROOT / 'requirements-colab.txt')], check=True)
print('İzole Python ortamı hazır.')

In [ ]:
# 4) Console'u başlat. GeminiKey varsa Colab Secrets'tan yalnızca sunucu sürecine verilir.
import os, time, urllib.request
from google.colab import output, userdata

PORT = 8765
env = os.environ.copy()
# Optional credentials are read one by one. A missing secret never blocks startup,
# and values are passed only to the private server process.
for secret_name in ('GeminiKey', 'PEXELS_API_KEY', 'PIXABAY_API_KEY'):
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None
    if secret_value:
        env[secret_name] = secret_value
env['YOUTUBE_AUTOMATION_USE_GPU'] = '1'
cmd = [str(PYTHON), '-m', 'creator_console', '--host', '127.0.0.1',
       '--port', str(PORT), '--storage-root', str(PROJECTS),
       '--archive-root', str(ARCHIVE), '--source-library-root',
       str(SOURCE_VIDEOS), '--no-env-file', '--quiet']
log_handle = open('/content/creator_console.log', 'w', encoding='utf-8')
console_process = subprocess.Popen(cmd, cwd=APP_ROOT, env=env,
                                   stdout=log_handle, stderr=subprocess.STDOUT)
for _ in range(60):
    if console_process.poll() is not None:
        raise RuntimeError(Path('/content/creator_console.log').read_text(encoding='utf-8')[-2000:])
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/health', timeout=1) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise TimeoutError('Creator Console 60 saniyede hazır olmadı.')
print('Creator Console hazır. Gemini:', 'yapılandırıldı' if env.get('GeminiKey') else 'eksik (planlama yine kullanılabilir)')
output.serve_kernel_port_as_iframe(PORT, height=900)

In [ ]:
# 5) İsteğe bağlı: sunucuyu kontrollü kapat. Drive'daki proje/artefaktlar silinmez.
if 'console_process' in globals() and console_process.poll() is None:
    console_process.terminate()
    console_process.wait(timeout=15)
if 'log_handle' in globals() and not log_handle.closed:
    log_handle.close()
print('Console durduruldu; Drive verileri korundu.')